In [ ]:
----------------------------------
1зад

In [ ]:
1. Дефинирање на состојбата
Состојбата во секој чекор ја дефинираме како торка (tuple) од два елементи: (lantern_position, left_side).

lantern_position: Стринг кој означува каде се наоѓа фенерот ('left' или 'right').

left_side: Сортирана торка (tuple) која ги содржи имињата на луѓето кои сè уште се наоѓаат на почетната (левата) страна. Користиме торка бидејќи таа е непроменлив објект (hashable), што ни овозможува да ја чуваме во множеството на посетени состојби (explored).

Почетна состојба: ('left', ('A', 'B', 'C', 'D'))

Крајна (целна) состојба: ('right', ())

In [ ]:
2. Разгранување на состојба: successors()
Разгранувањето зависи од моменталната позиција на фенерот:

Ако фенерот е лево ('left'): Се прават комбинации за преминување на 1 или 2 лица од левата кон десната страна. Избраните луѓе се отстрануваат од левата страна, фенерот преминува на 'right', а цената на акцијата е еднаква на времетраењето на побавниот човек што преминува (max(time1, time2)).

Ако фенерот е десно ('right'): Се прават комбинации за враќање на 1 или 2 лица од десната кон левата страна (луѓе кои не се во left_side). Тие се додаваат назад во торката на левата страна, фенерот преминува на 'left', а цената е времетраењето на побавниот меѓу нив.

In [ ]:
3. Избор на алгоритам за пребарување
За решавање на овој проблем најпрактичен алгоритам е Пребарување со униформна цена (Uniform Cost Search - UCS).

Причина: Проблемот бара изнаоѓање на патека со минимално вкупно време (минимална цена на патеката), при што самите акции имаат различна тежина (цена) во зависност од тоа кои луѓе се движат. Алгоритмите за слепо пребарување како BFS наоѓаат оптимален пат само кога сите акции имаат иста цена, додека DFS воопшто не гарантира оптималност. UCS гарантирано го наоѓа најоптималното решение (најкраткото време) бидејќи секогаш прво го разгранува јазолот со најмала акумулирана цена дотогаш, користејќи приоритетна опашка (heap).

In [ ]:
import heapq

# Генеричка класа за дефинирање на проблем како во лабораториските вежби
class Problem:
    def __init__(self, initial_state, goal_state=None):
        self.initial_state = initial_state
        self.goal_state = goal_state

    def successors(self, state):
        """Враќа листа од (акција, нова_состојба, цена_на_акција)"""
        pass

    def is_goal(self, state):
        return state == self.goal_state


# Специфична класа за проблемот со мостот и фенерот
class BridgeProblem(Problem):
    def __init__(self, people_times):
        """
        people_times: речник во формат {'A': 1, 'B': 2, ...}
        """
        self.times = people_times
        # Почетна состојба: (позиција_фенер, сортирана торка од луѓе лево)
        initial = ('left', tuple(sorted(self.times.keys())))
        # Целна состојба: фенерот е десно, лево нема никој
        goal = ('right', ())
        
        super().__init__(initial, goal)

    def successors(self, state):
        lantern, left_side = state
        left_list = list(left_side)
        # Сите луѓе кои во моментот преминале на десната страна
        right_list = [p for p in self.times.keys() if p not in left_side]
        
        successors_list = []
        
        if lantern == 'left':
            # Насока: Лево -> Десно (преминуваат 1 или 2 луѓе)
            # Случај 1: Преминува само 1 човек
            for i in range(len(left_list)):
                p1 = left_list[i]
                new_left = tuple(sorted([p for p in left_list if p != p1]))
                cost = self.times[p1]
                action = f"Преминува {p1}"
                successors_list.append((action, ('right', new_left), cost))
                
            # Случај 2: Преминуваат 2 луѓе заедно (цената е побавниот)
            for i in range(len(left_list)):
                for j in range(i + 1, len(left_list)):
                    p1, p2 = left_list[i], left_list[j]
                    new_left = tuple(sorted([p for p in left_list if p != p1 and p != p2]))
                    cost = max(self.times[p1], self.times[p2])
                    action = f"Преминуваат {p1} и {p2}"
                    successors_list.append((action, ('right', new_left), cost))
                    
        else: # lantern == 'right'
            # Насока: Десно -> Лево (враќање на фенерот со 1 или 2 луѓе)
            # Случај 1: Се враќа 1 човек со фенерот
            for i in range(len(right_list)):
                p1 = right_list[i]
                new_left = tuple(sorted(left_list + [p1]))
                cost = self.times[p1]
                action = f"Се враќа {p1}"
                successors_list.append((action, ('left', new_left), cost))
                
            # Случај 2: Се враќаат 2 луѓе (можен чекор, иако најчесто неоптимален)
            for i in range(len(right_list)):
                for j in range(i + 1, len(right_list)):
                    p1, p2 = right_list[i], right_list[j]
                    new_left = tuple(sorted(left_list + [p1, p2]))
                    cost = max(self.times[p1], self.times[p2])
                    action = f"Се враќаат {p1} и {p2}"
                    successors_list.append((action, ('left', new_left), cost))
                    
        return successors_list


# Алгоритам за пребарување со униформна цена (UCS)
def uniform_cost_search(problem):
    # Приоритетна опашка во формат: (вкупна_цена, моментална_состојба, патека_до_тука)
    queue = [(0, problem.initial_state, [])]
    explored = set()  # Множество за затворени/посетени состојби
    
    while queue:
        cost, state, path = heapq.heappop(queue)
        
        # Проверка за цел преку методот на класата Problem
        if problem.is_goal(state):
            return cost, path
            
        if state in explored:
            continue
            
        explored.add(state)
        
        # Разгранување преку методот successors()
        for action, next_state, action_cost in problem.successors(state):
            if next_state not in explored:
                heapq.heappush(queue, (cost + action_cost, next_state, path + [action]))
                
    return None, None


# ==========================================
# Извршување на задачите
# ==========================================

if __name__ == "__main__":
    # --- Задача 1: Конкретен тест пример со 4 лица ---
    print("=== РЕШЕНИЕ ЗА ОСНОВНАТА ЗАДАНА (4 лица) ===")
    people_task1 = {'A': 1, 'B': 2, 'C': 5, 'D': 10}
    bridge_problem_1 = BridgeProblem(people_task1)

    time_1, path_1 = uniform_cost_search(bridge_problem_1)

    print(f"Минимално вкупно време: {time_1} минути")
    print("Редослед на чекори:")
    for i, step in enumerate(path_1, 1):
        print(f"  {i}. {step}")

    # --- Задача 2: Тестирање на проширеното решение за произволен број на луѓе ---
    print("\n=== РЕШЕНИЕ ЗА ПРОШИРЕНАТА ЗАДАТА (Произволен број на луѓе) ===")
    # Пример со 5 лица со произволно зададени времиња
    people_task2 = {
        'Лице_1': 1,
        'Лице_2': 3,
        'Лице_3': 7,
        'Лице_4': 8,
        'Лице_5': 12
    }
    bridge_problem_2 = BridgeProblem(people_task2)

    time_2, path_2 = uniform_cost_search(bridge_problem_2)

    print(f"Минимално вкупно време: {time_2} минути")
    print("Редослед на чекори:")
    for i, step in enumerate(path_2, 1):
        print(f"  {i}. {step}")